# 6. The reference defense

Every baseline suppresses, and suppression is the signature. This one substitutes instead:
it answers from public information, confidently and in normal register, rather than
refusing.

On the GPU tiers that is logit arithmetic, `z = z_base + alpha * (z_retain - z_forget)`,
using two small adapters on one frozen base. Subtracting the forget expert removes the
marginal information the private documents added. Adding the retain expert supplies the
public answer that should be there instead.

In [ ]:
# On Kaggle or Colab, uncomment to install
# !pip install -q -e /kaggle/working/silentwall
# !pip install -q -e .

from silentwall.config import load_config
from silentwall.pipeline import prepare_workspace, run_method, run_sweep, save_workspace
from silentwall.report.render import render_comparison, render_markdown, write_comparison

CONFIG = "../configs/smoke.yaml"
cfg = load_config(CONFIG)
print(cfg.profile, cfg.tier, "methods:", len(cfg.methods))

In [ ]:
ws = prepare_workspace(cfg, verbose=False)

results = []
for method_id in ("clean_reference", "refusal_classifier", "silentwall"):
    results.append(run_method(ws, method_id))

In [ ]:
print(render_comparison(results))

## Calibration

Substituting a fluent answer is not enough. If the substitution is systematically shorter,
or never hedges, or repeats itself across samples while genuine answers vary, each of those
becomes a feature the detector reads.

So the defense measures the control distribution on the dev split and generates to match
it. Control entities only, dev split only, and the split tripwire raises if that pass ever
reaches an eval entity.

Two design notes worth knowing:

The defense applies its adjustment to every entity, restricted and control alike, doing
identical work in both cases. For controls the blend weight is zero, so the passes are
wasted. That waste is deliberate. Gating only on restricted entities would make those
queries measurably slower, and latency is one of the features the detector reads.

An earlier version dropped the token trace when substituting. Restricted entities then had
missing entropy features while controls had real ones, so availability of the feature
became a cleaner label than anything the feature measured, and AUC went to 1.0. Fixing one
side channel opened a louder one, which is the failure mode this whole project is about.

In [ ]:
sw = [r for r in results if r.method_id == "silentwall"][0]
print("leak by family:")
for lr in sw.leak:
    print(f"  {lr.family.value:20s} leak@1 {lr.leak_at_1.point:.3f}   leak@k {lr.leak_at_k.point:.3f}")
print()
det = sw.primary_detectability
print("detectability:", det.auc)